## Import Datas

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [ ]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [ ]:
import pandas as pd
from scipy.stats import norm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:


def heston_price(S, K, T, r, v0, kappa, theta, sigma, rho, option_type='call'):
    # Dates
    calendar = ql.NullCalendar()
    day_count = ql.Actual365Fixed()
    today = ql.Date.todaysDate()

    maturity_date = today + int(T * 365)

    # Processus
    spot = ql.QuoteHandle(ql.SimpleQuote(S))
    risk_free = ql.YieldTermStructureHandle(
        ql.FlatForward(today, r, day_count)
    )
    dividend = ql.YieldTermStructureHandle(
        ql.FlatForward(today, 0.0, day_count)
    )

    heston_process = ql.HestonProcess(
        risk_free, dividend, spot,
        v0, kappa, theta, sigma, rho
    )

    model = ql.HestonModel(heston_process)
    engine = ql.AnalyticHestonEngine(model)

    payoff = ql.PlainVanillaPayoff(
        ql.Option.Call if option_type == 'call' else ql.Option.Put,
        K
    )

    exercise = ql.EuropeanExercise(maturity_date)
    option = ql.VanillaOption(payoff, exercise)
    option.setPricingEngine(engine)

    return option.NPV()



In [ ]:
def compute_heston_price(ticker, option_type):
  data = dataset[ticker]["test"]
  data = data[data["optionType"] == option_type]

  test_data['Heston_price'] = test_data.apply(
    lambda row: heston_price(
        S=row['underlyingLastPrice'],
        K=row['strike'],
        T=row['timeToMaturity'],
        r=row['riskFreeRate'],
        v0=v0,
        kappa=kappa,
        theta=theta,
        sigma=sigma,
        rho=rho,
        option_type=row['optionType']
    ),
    axis=1
  )
  return test_data

## Heston price

In [ ]:
ticker = "MSFT"
option_type="put"

In [ ]:
compute_heston_price(ticker, option_type)